# MEN 4338 · Week 14 — Final project: resource estimate for Kossou

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/men4338/notebooks/MEN4338_W14_final_project_kossou.ipynb)

**IUGB · BSc Mining Engineering · Geostatistics (lecture & lab)** · Instructor: Laurent Kouadio

You are the geostatistician of a junior mining company. The Kossou gold prospect (synthetic) has been
drilled with RC holes on a 50 m grid, plus an infill pattern on the best zone. Each hole samples an **8 m
bench**. The board wants a **mineral resource estimate** at a cut-off of 0.8 g/t, classified into
Measured, Indicated and Inferred, before deciding on a feasibility study.

This notebook gives the steps and checks the consistency of your numbers. The geological choices (variogram
model, classification rule) are yours: justify them in the report.

**Deliverables:** this notebook (all checks passing) · a 6–8 page report · a 10-minute presentation.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/men4338/'
FILES = [('data/kossou_intervals.csv', 'kossou_intervals.csv'), ('data/kossou_collars.csv', 'kossou_collars.csv'), ('tests/w14/q1.py', 'tests/q1.py'), ('tests/w14/q2.py', 'tests/q2.py'), ('tests/w14/q3.py', 'tests/q3.py'), ('tests/w14/q4.py', 'tests/q4.py'), ('tests/w14/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

In [ ]:
# Toolbox from earlier weeks: run this cell.
import numpy as np

def exp_variogram(x, y, v, lag, nlags, azimuth=None, atol=22.5):
    """Experimental semivariogram. Lag class k (1..nlags) holds the pairs with |d - k*lag| < lag/2.
    With an azimuth (degrees clockwise from north), only pairs within +/- atol of that direction are kept.
    Returns mean distance, semivariance and number of pairs of each class (NaN where a class is empty)."""
    x, y, v = (np.asarray(a, float) for a in (x, y, v))
    i, j = np.triu_indices(len(v), 1)
    dx, dy = x[j] - x[i], y[j] - y[i]
    d = np.hypot(dx, dy)
    keep = np.ones(len(d), bool)
    if azimuth is not None:
        az = np.degrees(np.arctan2(dx, dy)) % 180
        diff = np.abs(az - azimuth % 180)
        keep = np.minimum(diff, 180 - diff) <= atol
    sq = 0.5 * (v[j] - v[i]) ** 2
    hm, g, n = np.full(nlags, np.nan), np.full(nlags, np.nan), np.zeros(nlags, int)
    for k in range(1, nlags + 1):
        m = keep & (np.abs(d - k * lag) < lag / 2)
        n[k - 1] = m.sum()
        if n[k - 1]:
            hm[k - 1], g[k - 1] = d[m].mean(), sq[m].mean()
    return hm, g, n

def spherical(h, c0, c, a):
    """Spherical model: nugget c0, partial sill c, range a. gamma(0) = 0."""
    h = np.asarray(h, float)
    r = np.minimum(h / a, 1.0)
    return np.where(h > 0, c0 + c * (1.5 * r - 0.5 * r ** 3), 0.0)

def aniso_h(dx, dy, azimuth, a_major, a_minor):
    """Distance rescaled to the major range: an anisotropic model becomes model(aniso_h(...), c0, c, a_major)."""
    t = np.radians(azimuth)
    u = dx * np.sin(t) + dy * np.cos(t)
    w = dx * np.cos(t) - dy * np.sin(t)
    return np.sqrt(u ** 2 + (w * a_major / a_minor) ** 2)

def cell_weights(x, y, cell_size):
    """Cell declustering weights (sum to 1); cells start at the origin."""
    key = np.floor(np.asarray(x) / cell_size).astype(int) * 100000 + np.floor(np.asarray(y) / cell_size).astype(int)
    _, inv, cnt = np.unique(key, return_inverse=True, return_counts=True)
    w = 1.0 / cnt[inv]
    return w / w.sum()

def ok_point(x0, y0, x, y, v, gamma, nmax=16):
    """Ordinary kriging at (x0, y0) with the nmax nearest data.
    gamma(dx, dy) is the variogram model, a function of the separation vector.
    Returns (estimate, kriging variance)."""
    d = np.hypot(x - x0, y - y0)
    idx = np.argsort(d)[:nmax]
    xs, ys, vs = x[idx], y[idx], v[idx]
    n = len(idx)
    A = np.ones((n + 1, n + 1)); A[n, n] = 0
    A[:n, :n] = gamma(xs[:, None] - xs[None, :], ys[:, None] - ys[None, :])
    b = np.ones(n + 1); b[:n] = gamma(xs - x0, ys - y0)
    sol = np.linalg.solve(A, b)
    w, mu = sol[:n], sol[n]
    return float(w @ vs), float(w @ b[:n] + mu)

def discretise(xc, yc, size, n):
    """n x n points at the centres of the sub-cells of a square block of side size centred on (xc, yc)."""
    o = (np.arange(n) + 0.5) / n * size - size / 2
    px, py = np.meshgrid(xc + o, yc + o)
    return px.ravel(), py.ravel()

def gamma_bar(gamma, size, n=5):
    """Average variogram within a square block."""
    px, py = discretise(0, 0, size, n)
    return float(gamma(px[:, None] - px[None, :], py[:, None] - py[None, :]).mean())

def ok_block(xc, yc, size, x, y, v, gamma, nd=4, nmax=16):
    """Ordinary block kriging of a size x size block centred on (xc, yc). Returns (estimate, block kriging variance)."""
    k = np.argsort(np.hypot(x - xc, y - yc))[:nmax]
    xs, ys = x[k], y[k]; m = len(k)
    px, py = discretise(xc, yc, size, nd)
    A = np.ones((m + 1, m + 1)); A[m, m] = 0
    A[:m, :m] = gamma(xs[:, None] - xs[None, :], ys[:, None] - ys[None, :])
    b = np.ones(m + 1); b[:m] = gamma(xs[:, None] - px[None, :], ys[:, None] - py[None, :]).mean(axis=1)
    s = np.linalg.solve(A, b)
    return float(s[:m] @ v[k]), float(s[:m] @ b[:m] + s[m] - gamma_bar(gamma, size, nd))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

intervals = pd.read_csv("kossou_intervals.csv")
collars = pd.read_csv("kossou_collars.csv")
print(intervals.shape, collars.shape)

## Project steps

| step | weeks | output |
|---|---|---|
| 1. Composites and data check | 1–2 | `ks`, location map, histogram |
| 2. Clustering and declustered statistics | 2 | `is_infill`, declustered mean |
| 3. Variography and model | 3–5 | directional variograms, `ks_model`, `gamma_ks` |
| 4. Validation | 6, 9 | cross-validation (mean standardised squared error near 1) |
| 5. Block model | 8–10 | 50 m blocks, estimates and block kriging variances |
| 6. Classification and resource table | 10 | Measured / Indicated / Inferred at 0.8 g/t |

**Classification.** A common rule uses the block kriging variance (or the distance to the data): blocks with a
low variance are **Measured**, intermediate **Indicated**, the rest **Inferred**. Choose two thresholds,
e.g. from the variance of blocks fully inside the infill pattern and inside the 50 m grid, and justify them.

---
## Checked steps

### Q1. Composites

Build `ks` (columns `hole_id`, `x`, `y`, `au_gpt`: the 8 m bench composite of each hole). Store `n_holes` and
`sampled_m` (total length of assayed intervals).

In [ ]:
ks = ...
n_holes = ...
sampled_m = ...
n_holes, sampled_m

In [ ]:
grader.check("q1")

### Q2. Clustering and declustering

Add a boolean column `ks["is_infill"]`: True when the nearest other hole is closer than 40 m. Store the plain
mean in `ks_naive_mean` and the declustered mean with 50 m cells in `ks_declus_mean`.

In [ ]:
x, y, au = ks["x"].to_numpy(), ks["y"].to_numpy(), ks["au_gpt"].to_numpy()
ks["is_infill"] = ...
ks_naive_mean = ...
ks_declus_mean = ...
ks_naive_mean, ks_declus_mean

In [ ]:
grader.check("q2")

### Q3. Your variogram model

Study the directional variograms of the **grid** holes (`~ks["is_infill"]`), choose an anisotropic spherical
model and store it in `ks_model = {"c0": ..., "c": ..., "azimuth": ..., "a_major": ..., "a_minor": ...}`.
Define `gamma_ks(dx, dy)` from it. The check cross-validates your model on all holes (16 neighbours): the mean
standardised squared error must be between 0.7 and 1.4.

In [ ]:
ks_model = {"c0": ..., "c": ..., "azimuth": ..., "a_major": ..., "a_minor": ...}

def gamma_ks(dx, dy):
    ...

In [ ]:
grader.check("q3")

### Q4. Block model

Krig the 50 m blocks (16 rows × 20 columns, block `(r, c)` centred on `(25 + 50*c, 25 + 50*r)`) with
`ok_block(..., 50, x, y, au, gamma_ks)`. Store `ks_blocks` and `ks_block_var`. Map both.

In [ ]:
ks_blocks = np.zeros((16, 20))
ks_block_var = np.zeros((16, 20))
...

In [ ]:
grader.check("q4")

### Q5. Resource table

Classify the blocks with your two variance thresholds and build the DataFrame `resource` with the rows
`Measured`, `Indicated`, `Inferred` (in that order) and the columns `category`, `tonnes_mt`, `grade_gpt`,
`metal_koz`, for the blocks estimated ≥ 0.8 g/t (54 000 t per block, 31.1035 g per ounce).

In [ ]:
t_measured = ...
t_indicated = ...
resource = ...
resource

In [ ]:
grader.check("q5")

### For the report (not checked)

* Grade–tonnage curves for cut-offs 0 to 2 g/t, and the sensitivity of the resource to the cut-off.
* A sequential Gaussian simulation (week 13) of the bench: P90 / P50 / P10 of the tonnage above 0.8 g/t.
  Is the Measured + Indicated tonnage inside that range?
* Risks and recommendations: where would you drill next, and why?

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()